## Step 0: Suppress Warnings (Optional)
Run this first for cleaner output

In [ ]:
# Suppress warnings for cleaner output
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print("✓ Environment configured for clean output")

## Step 1: Clone Repository

In [1]:
!git clone https://github.com/ahmedelbamby-aast/DeepFakeBenchUpgraded.git

Cloning into 'DeepFakeBenchUpgraded'...


## Step 2: Change Directory

In [2]:
%cd DeepFakeBenchUpgraded

d:\Computer Vision Project\DeepfakeBench\DeepFakeBenchUpgraded


## Step 3: Install DeepfakeBench
This takes ~2-3 minutes. Dependency warnings are normal and can be ignored.

In [3]:
!bash kaggle_install.sh

w s l :   A n   e r r o r   o c c u r r e d   m o u n t i n g   t h e   d i s t r i b u t i o n   d i s k ,   i t   w a s   m o u n t e d   r e a d - o n l y   a s   a   f a l l b a c k . 
 
 w s l :   S e e   r e c o v e r y   i n s t r u c t i o n s   o n :   h t t p s : / / a k a . m s / w s l d i s k m o u n t r e c o v e r y 
 
 <3>WSL (10 - Relay) ERROR: CreateProcessCommon:800: execvpe(/bin/bash) failed: No such file or directory


## Step 4: Verify Installation
Test if everything works correctly

In [ ]:
import torch
import sys

# Check PyTorch and GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Test DeepfakeBench import
sys.path.insert(0, '.')
from deepfakebench.detectors.xception_detector import XceptionDetector
print("\n✅ DeepfakeBench is ready to use!")

## Optional: Quick Model Test
Test loading a detector model

In [ ]:
import yaml

# Load Xception config
with open('deepfakebench/config/detector/xception.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Config loaded successfully!")
print(f"Detector: {config.get('detector_name', 'Unknown')}")
print(f"Backbone: {config.get('backbone_name', 'Unknown')}")

## Next Steps

### Training
```bash
!python deepfakebench/train.py \
    --detector_path ./deepfakebench/config/detector/xception.yaml \
    --train_dataset FF-DF \
    --test_dataset FF-DF
```

### Testing
```bash
!python deepfakebench/test.py \
    --detector_path ./deepfakebench/config/detector/xception.yaml \
    --test_dataset FF-DF \
    --weights_path /path/to/checkpoint.pth
```

### Add Datasets
1. Go to **Add Data** → Search for "FaceForensics", "Celeb-DF", or "DFDC"
2. Add to notebook
3. Link datasets:
```bash
!mkdir -p datasets/rgb
!ln -s /kaggle/input/your-dataset datasets/rgb/FaceForensics++
```

## Test 6: Verify Kaggle Dataset Structure Compatibility

In [ ]:
"""
This cell verifies your Kaggle dataset structure is compatible with DeepfakeBench.

Your dataset structure:
/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++/
├── manipulated_sequences/
│   ├── Face2Face/, Deepfakes/, DeepFakeDetection/, NeuralTextures/, FaceShifter/, FaceSwap/
│   │   └── c23/frames/[video_folders]/[frame_files]
└── original_sequences/
    └── youtube/c23/frames/[video_folders]/[frame_files]
"""

import os
import sys

# Simulate Kaggle path structure locally
base_path_kaggle = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++'

# For local testing, use a mock structure
print("Testing dataset structure compatibility...")
print("=" * 70)

# Expected structure
expected_structure = {
    'manipulated_sequences': ['Face2Face', 'Deepfakes', 'DeepFakeDetection', 
                              'NeuralTextures', 'FaceShifter', 'FaceSwap'],
    'original_sequences': ['youtube']
}

print("\n✅ Expected Structure:")
print(f"  - manipulated_sequences/ with methods: {expected_structure['manipulated_sequences']}")
print(f"  - original_sequences/ with: {expected_structure['original_sequences']}")
print(f"  - Each method has: c23/frames/[video_folders]/[frame_files]")

print("\n✅ System Support:")
print("  - preprocessing/rearrange.py: Scans this exact structure")
print("  - dataset/abstract_dataset.py: Loads from JSON mappings")
print("  - Compression c23: Fully supported")

print("\n✅ Configuration for Kaggle:")
print(f"""
config = {{
    'rgb_dir': '{base_path_kaggle.replace('FaceForensics++', '')}',
    'dataset_json_folder': './preprocessing/dataset_json',
    'compression': 'c23',
    'train_dataset': ['FaceForensics++'],  # or ['FF-F2F', 'FF-DF', etc.]
    'test_dataset': 'FaceForensics++'
}}
""")

print("\n📋 Next Steps on Kaggle:")
print("  1. Run rearrange.py to generate JSON mapping")
print("  2. Update config with your dataset path")
print("  3. Start training or testing")
print("\nSee KAGGLE_DATASET_GUIDE.md for complete instructions!")
print("=" * 70)

## Test 7: Generate Dataset JSON Mapping
This step scans your dataset structure and creates JSON files for training/testing

In [ ]:
import os
import json
from pathlib import Path

def scan_dataset_structure(dataset_root, dataset_name='FaceForensics++'):
    """
    Comprehensive scanner for FaceForensics++ dataset structure.
    Reports what exists and what's missing.
    """
    
    print("=" * 80)
    print("DATASET STRUCTURE SCANNER")
    print("=" * 80)
    print(f"\nScanning: {dataset_root}")
    print(f"Dataset: {dataset_name}\n")
    
    dataset_path = os.path.join(dataset_root, dataset_name)
    
    # Results tracking
    results = {
        'required_found': [],
        'required_missing': [],
        'optional_found': [],
        'optional_missing': [],
        'unexpected_found': []
    }
    
    # ========== 1. SPLIT FILES (REQUIRED) ==========
    print("📁 SPLIT FILES (Train/Val/Test)")
    print("-" * 80)
    
    split_files = ['train.json', 'val.json', 'test.json']
    split_locations = [
        dataset_path,
        '/kaggle/working/splits',
        './splits'
    ]
    
    split_found = False
    for split_file in split_files:
        found_at = None
        for location in split_locations:
            path = os.path.join(location, split_file)
            if os.path.exists(path):
                found_at = location
                break
        
        if found_at:
            results['required_found'].append(f"✓ {split_file} → {found_at}")
            print(f"  ✓ {split_file:<15} FOUND at {found_at}")
            split_found = True
            
            # Show file size and sample content
            try:
                with open(os.path.join(found_at, split_file), 'r') as f:
                    data = json.load(f)
                    print(f"     └─ Contains {len(data)} video pairs")
            except:
                pass
        else:
            results['required_missing'].append(f"✗ {split_file}")
            print(f"  ✗ {split_file:<15} MISSING (checked: {', '.join(split_locations)})")
    
    if not split_found:
        print("\n  ⚠️  WARNING: No split files found. These are REQUIRED for training.")
        print("     Will be auto-generated on first run to /kaggle/working/splits/")
    
    # ========== 2. ORIGINAL SEQUENCES (REAL VIDEOS) ==========
    print("\n\n📹 ORIGINAL SEQUENCES (Real Videos)")
    print("-" * 80)
    
    orig_seq_path = os.path.join(dataset_path, 'original_sequences')
    
    # YouTube real videos (REQUIRED)
    youtube_structures = [
        ('youtube/c23/videos', 'REQUIRED', 'YouTube real videos (c23 compression, video files)'),
        ('youtube/c23/frames', 'OPTIONAL', 'YouTube real videos (c23 compression, extracted frames)'),
        ('youtube/c40/videos', 'OPTIONAL', 'YouTube real videos (c40 compression, video files)'),
        ('youtube/c40/frames', 'OPTIONAL', 'YouTube real videos (c40 compression, extracted frames)'),
        ('youtube/raw/videos', 'OPTIONAL', 'YouTube real videos (raw, uncompressed video files)'),
        ('youtube/raw/frames', 'OPTIONAL', 'YouTube real videos (raw, extracted frames)'),
    ]
    
    for subpath, requirement, description in youtube_structures:
        full_path = os.path.join(orig_seq_path, subpath)
        exists = os.path.exists(full_path)
        
        if exists:
            # Count files/directories
            try:
                items = os.listdir(full_path)
                count = len([i for i in items if os.path.isfile(os.path.join(full_path, i)) or os.path.isdir(os.path.join(full_path, i))])
                status = f"✓ {count} items"
                
                if requirement == 'REQUIRED':
                    results['required_found'].append(f"✓ {subpath} ({count} items)")
                else:
                    results['optional_found'].append(f"✓ {subpath} ({count} items)")
                    
                print(f"  ✓ {subpath:<35} {status:<20} [{requirement}]")
                print(f"     └─ {description}")
            except:
                print(f"  ✓ {subpath:<35} EXISTS (can't count) [{requirement}]")
        else:
            if requirement == 'REQUIRED':
                results['required_missing'].append(f"✗ {subpath}")
                print(f"  ✗ {subpath:<35} MISSING             [{requirement}]")
            else:
                results['optional_missing'].append(f"✗ {subpath}")
                print(f"  - {subpath:<35} Not present         [{requirement}]")
    
    # DFD actors (OPTIONAL)
    print("\n  DFD Real (Actors Dataset - OPTIONAL):")
    actors_structures = [
        ('actors/c23/videos', 'OPTIONAL'),
        ('actors/c23/frames', 'OPTIONAL'),
        ('actors/c40/videos', 'OPTIONAL'),
        ('actors/c40/frames', 'OPTIONAL'),
    ]
    
    for subpath, requirement in actors_structures:
        full_path = os.path.join(orig_seq_path, subpath)
        if os.path.exists(full_path):
            try:
                count = len(os.listdir(full_path))
                results['optional_found'].append(f"✓ {subpath} ({count} items)")
                print(f"    ✓ {subpath:<33} {count} items")
            except:
                print(f"    ✓ {subpath:<33} EXISTS")
        else:
            results['optional_missing'].append(f"- {subpath}")
            print(f"    - {subpath:<33} Not present")
    
    # ========== 3. MANIPULATED SEQUENCES (FAKE VIDEOS) ==========
    print("\n\n🎭 MANIPULATED SEQUENCES (Fake Videos)")
    print("-" * 80)
    
    manip_seq_path = os.path.join(dataset_path, 'manipulated_sequences')
    
    if os.path.exists(manip_seq_path):
        results['required_found'].append(f"✓ manipulated_sequences directory")
        print(f"  ✓ manipulated_sequences directory exists\n")
        
        # Expected manipulation methods
        expected_methods = [
            ('Deepfakes', 'OPTIONAL', 'Face swap using autoencoders'),
            ('Face2Face', 'OPTIONAL', 'Expression transfer'),
            ('FaceSwap', 'OPTIONAL', 'Classic face swap method'),
            ('NeuralTextures', 'OPTIONAL', 'Neural rendering-based'),
            ('FaceShifter', 'OPTIONAL', 'High-quality face swap'),
            ('DeepFakeDetection', 'OPTIONAL', 'DFD dataset methods'),
        ]
        
        try:
            methods = [d for d in os.listdir(manip_seq_path) if os.path.isdir(os.path.join(manip_seq_path, d))]
            
            for method, requirement, description in expected_methods:
                if method in methods:
                    method_path = os.path.join(manip_seq_path, method)
                    
                    # Check compression levels
                    compressions = []
                    for comp in ['c23', 'c40', 'raw']:
                        comp_path = os.path.join(method_path, comp)
                        if os.path.exists(comp_path):
                            # Check for videos or frames
                            has_videos = os.path.exists(os.path.join(comp_path, 'videos'))
                            has_frames = os.path.exists(os.path.join(comp_path, 'frames'))
                            
                            if has_videos:
                                try:
                                    count = len(os.listdir(os.path.join(comp_path, 'videos')))
                                    compressions.append(f"{comp}/videos ({count})")
                                except:
                                    compressions.append(f"{comp}/videos")
                            if has_frames:
                                try:
                                    count = len(os.listdir(os.path.join(comp_path, 'frames')))
                                    compressions.append(f"{comp}/frames ({count})")
                                except:
                                    compressions.append(f"{comp}/frames")
                    
                    if compressions:
                        results['optional_found'].append(f"✓ {method}")
                        print(f"  ✓ {method:<25} {', '.join(compressions)}")
                        print(f"     └─ {description}")
                    else:
                        print(f"  ✓ {method:<25} (directory exists but no c23/c40/raw found)")
                else:
                    results['optional_missing'].append(f"- {method}")
                    print(f"  - {method:<25} Not present [{requirement}]")
            
            # Check for unexpected methods
            known_methods = [m[0] for m in expected_methods]
            unexpected = [m for m in methods if m not in known_methods]
            if unexpected:
                print(f"\n  ℹ️  Unexpected methods found: {', '.join(unexpected)}")
                for method in unexpected:
                    results['unexpected_found'].append(f"? {method}")
                    
        except Exception as e:
            print(f"  ⚠️  Error scanning manipulated_sequences: {e}")
    else:
        results['required_missing'].append(f"✗ manipulated_sequences directory")
        print(f"  ✗ manipulated_sequences directory NOT FOUND")
        print(f"     This is optional but recommended for training on fake videos")
    
    # ========== 4. METADATA & CONFIG FILES ==========
    print("\n\n📄 METADATA & CONFIGURATION FILES")
    print("-" * 80)
    
    metadata_files = [
        ('dataset_json/FaceForensics++.json', 'OPTIONAL', './deepfakebench/preprocessing/dataset_json/', 'Generated dataset mapping'),
    ]
    
    for filename, requirement, base_path, description in metadata_files:
        # Check multiple locations
        locations = [
            os.path.join(dataset_path, filename),
            os.path.join('.', filename),
            os.path.join(base_path, os.path.basename(filename)),
        ]
        
        found = False
        for loc in locations:
            if os.path.exists(loc):
                file_size = os.path.getsize(loc)
                size_str = f"{file_size:,} bytes" if file_size < 1024*1024 else f"{file_size/(1024*1024):.2f} MB"
                results['optional_found'].append(f"✓ {filename}")
                print(f"  ✓ {filename:<45} {size_str}")
                print(f"     └─ {description}")
                print(f"     └─ Location: {loc}")
                found = True
                break
        
        if not found:
            results['optional_missing'].append(f"- {filename}")
            print(f"  - {filename:<45} Not found [{requirement}]")
            print(f"     └─ {description} (will be generated)")
    
    # ========== 5. SUMMARY ==========
    print("\n\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    
    print(f"\n✓ REQUIRED Files Found: {len(results['required_found'])}")
    for item in results['required_found'][:5]:
        print(f"  {item}")
    if len(results['required_found']) > 5:
        print(f"  ... and {len(results['required_found']) - 5} more")
    
    print(f"\n✗ REQUIRED Files Missing: {len(results['required_missing'])}")
    if results['required_missing']:
        for item in results['required_missing']:
            print(f"  {item}")
    else:
        print(f"  None - all required files present!")
    
    print(f"\n✓ OPTIONAL Files Found: {len(results['optional_found'])}")
    for item in results['optional_found'][:3]:
        print(f"  {item}")
    if len(results['optional_found']) > 3:
        print(f"  ... and {len(results['optional_found']) - 3} more")
    
    print(f"\n- OPTIONAL Files Missing: {len(results['optional_missing'])}")
    if len(results['optional_missing']) <= 5:
        for item in results['optional_missing']:
            print(f"  {item}")
    else:
        print(f"  {len(results['optional_missing'])} optional components not present")
    
    if results['unexpected_found']:
        print(f"\n? UNEXPECTED Files Found: {len(results['unexpected_found'])}")
        for item in results['unexpected_found']:
            print(f"  {item}")
    
    # ========== 6. RECOMMENDATIONS ==========
    print("\n\n" + "=" * 80)
    print("RECOMMENDATIONS")
    print("=" * 80 + "\n")
    
    if results['required_missing']:
        print("⚠️  REQUIRED ITEMS MISSING:")
        for item in results['required_missing']:
            if 'train.json' in item or 'val.json' in item or 'test.json' in item:
                print(f"  • {item}")
                print(f"     → Will be auto-generated to /kaggle/working/splits/")
            elif 'youtube/c23' in item:
                print(f"  • {item}")
                print(f"     → This is essential for training. Check your dataset upload.")
            else:
                print(f"  • {item}")
    else:
        print("✓ All required files present!")
    
    if not results['optional_found']:
        print("\nℹ️  No optional components found:")
        print("  • No manipulated sequences (fake videos)")
        print("  → You can only train on real vs other dataset's fake")
        print("  → Consider adding manipulation methods for better training")
    
    print("\n" + "=" * 80)
    print("Scan complete!")
    print("=" * 80)
    
    return results

# Run the scanner
# Change these paths based on your environment
if os.path.exists('/kaggle/input'):
    # Kaggle environment
    dataset_root = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb'
else:
    # Local environment
    dataset_root = './datasets/rgb'

results = scan_dataset_structure(dataset_root, 'FaceForensics++')

## Test 6.5: Dataset Structure Scanner

Scan your dataset to identify what files and directories exist vs what's required/optional

**⚠️ Kaggle Environment Notes:**

1. **Read-only filesystem**: On Kaggle, `/kaggle/input/` is **read-only**. If your dataset doesn't include `train.json`, `val.json`, and `test.json` files, they will be automatically created in `/kaggle/working/splits/` instead.

2. **Videos vs Frames**: The code now automatically detects whether your dataset uses `videos/` or `frames/` directories and handles both structures.

The code has been updated to automatically:
- Search for split files in: Dataset directory → `/kaggle/working/splits/` → `./splits/`
- Detect and use either `videos/` or `frames/` directory structure

This fixes the `OSError: [Errno 30] Read-only file system` and `FileNotFoundError: No such file or directory: .../frames` errors.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/DeepFakeBenchUpgraded')

# Check if dataset exists
import os
dataset_path = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb/FaceForensics++'

if os.path.exists(dataset_path):
    print("✓ Dataset found!")
    print(f"  Path: {dataset_path}")
    
    # Check structure
    if os.path.exists(os.path.join(dataset_path, 'manipulated_sequences')):
        methods = os.listdir(os.path.join(dataset_path, 'manipulated_sequences'))
        print(f"  Manipulation methods: {methods}")
    
    if os.path.exists(os.path.join(dataset_path, 'original_sequences')):
        print(f"  Original sequences: Found")
    
    print("\nGenerating JSON mapping...")
    
    # Import and run rearrange
    from deepfakebench.preprocessing.rearrange import generate_dataset_file
    import os
    import json
    import glob
    
    # Create output directory
    os.makedirs('./deepfakebench/preprocessing/dataset_json', exist_ok=True)
    
    # Pass parent directory - function adds 'FaceForensics++' internally
    dataset_root = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb'
    
    # Check if train/val/test split JSONs exist
    split_files = ['train.json', 'val.json', 'test.json']
    split_path = os.path.join(dataset_root, 'FaceForensics++')
    
    # Since /kaggle/input is read-only, we'll create splits in /kaggle/working
    working_split_path = '/kaggle/working/splits'
    os.makedirs(working_split_path, exist_ok=True)
    
    missing_splits = [f for f in split_files if not os.path.exists(os.path.join(split_path, f))]
    
    if missing_splits:
        print(f"  ⚠ Missing split files in dataset: {missing_splits}")
        print(f"  Creating default 80/10/10 split in /kaggle/working/splits/...")
        
        # Get all video IDs from original sequences
        orig_path = os.path.join(dataset_path, 'original_sequences/youtube/c23/videos')
        if os.path.exists(orig_path):
            # Support both .mp4 and .avi files
            all_files = os.listdir(orig_path)
            video_files = [f for f in all_files if f.endswith(('.mp4', '.avi'))]
            video_ids = [os.path.splitext(f)[0] for f in video_files]
            
            if not video_ids:
                print(f"  ✗ No video files found in {orig_path}")
                print(f"     Checked for: .mp4 and .avi files")
            else:
                print(f"  ✓ Found {len(video_ids)} videos")
                
                # Create 80/10/10 split
                n = len(video_ids)
                train_end = int(0.8 * n)
                val_end = int(0.9 * n)
                
                train_pairs = [[v, v] for v in video_ids[:train_end]]
                val_pairs = [[v, v] for v in video_ids[train_end:val_end]]
                test_pairs = [[v, v] for v in video_ids[val_end:]]
                
                # Save split files to writable location
                with open(os.path.join(working_split_path, 'train.json'), 'w') as f:
                    json.dump(train_pairs, f)
                with open(os.path.join(working_split_path, 'val.json'), 'w') as f:
                    json.dump(val_pairs, f)
                with open(os.path.join(working_split_path, 'test.json'), 'w') as f:
                    json.dump(test_pairs, f)
                
                print(f"  ✓ Created splits: {len(train_pairs)} train, {len(val_pairs)} val, {len(test_pairs)} test")
                print(f"  ✓ Splits saved to: {working_split_path}")
                
                print(f"\n  ℹ Note: Split files created in {working_split_path}")
                print(f"     These will be used by the training script")
        else:
            print(f"  ✗ Cannot create splits: {orig_path} not found")
    else:
        print("  ✓ Split files already exist in dataset")
    
    # For the generate_dataset_file function, we'll still use the original dataset_root
    # but the split files will be read from /kaggle/working/splits if needed
    generate_dataset_file(
        dataset_name='FaceForensics++',
        dataset_root_path=dataset_root,
        output_file_path='./deepfakebench/preprocessing/dataset_json/FaceForensics++.json',
        compression_level='c23'
    )
    
    print("\n✅ JSON mapping generated successfully!")
    print("   Location: ./deepfakebench/preprocessing/dataset_json/FaceForensics++.json")
    print("\n⚠️  Important: If train/val/test.json were missing from the dataset,")
    print("    you'll need to copy them from /kaggle/working/splits/ to your dataset")
    print("    for the next run, OR use them directly from /kaggle/working/splits/")
else:
    print("⚠️ Dataset not found at:", dataset_path)
    print("\nFor local testing, this is expected.")
    print("On Kaggle, make sure to:")
    print("  1. Add the dataset to your notebook")
    print("  2. Verify the path matches your dataset input")
    print("  3. Ensure train.json, val.json, test.json exist in the dataset root")
    print("     (These files define the train/val/test split)")

## Test 8: Configure Training
Set up training configuration for your dataset

**⚠️ Important:** Run the cells in this order:
1. **Test 7** - Generate Dataset JSON Mapping (creates the JSON file)
2. **Test 8** - Configure Training (this cell - sets up config)
3. **Test 9** - Start Training (runs the training script)

If you get `dataset FaceForensics++ not exist!` error, make sure you've run Test 7 first!

In [ ]:
import yaml
import os

# Load base configuration
config_path = 'deepfakebench/config/detector/sladd_detector.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Kaggle
config['rgb_dir'] = '/kaggle/input/faceforensicsplusplus-c23-deepfakebench-structure/rgb'
config['dataset_json_folder'] = './deepfakebench/preprocessing/dataset_json'
config['log_dir'] = './kaggle_logs'

# Training settings
config['compression'] = 'c23'
config['train_dataset'] = ['FaceForensics++']  # Train on all methods
config['test_dataset'] = 'FaceForensics++'     # Test on all methods

# Optional: Train on specific methods only
# config['train_dataset'] = ['FF-F2F', 'FF-DF']  # Face2Face and Deepfakes only
# config['test_dataset'] = 'FF-FS'               # Test on FaceSwap

# Training hyperparameters (adjust as needed)
config['nEpochs'] = 5  # Number of epochs (increase for better results)
config['train_batchSize'] = 16  # Batch size (adjust based on GPU memory)
config['test_batchSize'] = 32
config['save_epoch'] = 1  # Save checkpoint every epoch

# Frame settings
config['frame_num'] = {'train': 8, 'test': 32}  # Frames per video
config['resolution'] = 256  # Image resolution

print("✅ Training Configuration:")
print(f"  Dataset path: {config['rgb_dir']}")
print(f"  JSON folder: {config['dataset_json_folder']}")
print(f"  Compression: {config['compression']}")
print(f"  Train datasets: {config['train_dataset']}")
print(f"  Test dataset: {config['test_dataset']}")
print(f"  Epochs: {config['nEpochs']}")
print(f"  Batch size: {config['train_batchSize']}")
print(f"  Frames per video: {config['frame_num']['train']}")

# Save configuration
kaggle_config_path = 'kaggle_training_config.yaml'
with open(kaggle_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"\n✅ Config saved to: {kaggle_config_path}")

# Verify the JSON file exists
json_file = os.path.join(config['dataset_json_folder'], 'FaceForensics++.json')
if os.path.exists(json_file):
    print(f"✅ Dataset JSON found: {json_file}")
else:
    print(f"⚠️  WARNING: Dataset JSON not found at {json_file}")
    print("   Make sure to run the 'Generate Dataset JSON Mapping' cell first!")

print("\nReady for training!")

## Test 9: Start Training
Run training with your configured settings

**⚠️ Path Fixes Applied:**
- All hardcoded paths (`./training/config/`) have been updated to use the correct structure (`deepfakebench/config/`)
- Scripts now use dynamic path resolution relative to their location
- Compatible with both Kaggle and local environments

**Note:** Training will take time depending on:
- Number of epochs
- Dataset size  
- GPU availability (T4 x2 recommended)

For a quick test, use 1-2 epochs. For actual training, use 20-40 epochs.

In [ ]:
# Option 1: Quick Training Test (1 epoch for testing)
# Uncomment to run a quick test
"""
!python deepfakebench/train.py \
    --detector_path kaggle_training_config.yaml \
    --train_dataset FaceForensics++ \
    --test_dataset FaceForensics++
"""

# Option 2: Full Training (configure epochs in the config above)
# This will train the model on your dataset
# Run this when you're ready for actual training:

print("Training commands:")
print("\n1. Quick test (already configured in config):")
print("   !python deepfakebench/train.py --detector_path kaggle_training_config.yaml")

print("\n2. Resume from checkpoint:")
print("   !python deepfakebench/train.py --detector_path kaggle_training_config.yaml --resume /path/to/checkpoint.pth")

print("\n3. Monitor training:")
print("   Check './kaggle_logs' directory for:")
print("   - Training logs")
print("   - Model checkpoints")
print("   - TensorBoard logs (if installed)")

print("\n⚠️ Important:")
print("  - Training time: ~30-60 min per epoch on T4 x2 GPU")
print("  - Make sure GPU is enabled in Kaggle settings")
print("  - Checkpoints saved every epoch to kaggle_logs/")

# Uncomment the line below when ready to train
# !python deepfakebench/train.py --detector_path kaggle_training_config.yaml

## Test 10: Monitor Training Progress (Optional)

Check training logs and saved checkpoints

In [ ]:
import os
import glob

# Check if training has started
log_dir = './kaggle_logs'

if os.path.exists(log_dir):
    print("✓ Training logs directory found")
    
    # List all subdirectories (each training run)
    runs = [d for d in os.listdir(log_dir) if os.path.isdir(os.path.join(log_dir, d))]
    
    if runs:
        print(f"\nTraining runs found: {len(runs)}")
        for run in runs:
            run_path = os.path.join(log_dir, run)
            print(f"\n  Run: {run}")
            
            # Check for checkpoints
            ckpts = glob.glob(os.path.join(run_path, '*.pth'))
            if ckpts:
                print(f"    Checkpoints: {len(ckpts)}")
                for ckpt in ckpts[:3]:  # Show first 3
                    print(f"      - {os.path.basename(ckpt)}")
            
            # Check for log files
            logs = glob.glob(os.path.join(run_path, '*.log'))
            if logs:
                print(f"    Log files: {len(logs)}")
                
                # Show last few lines of latest log
                latest_log = max(logs, key=os.path.getctime)
                print(f"\n    Latest log excerpt ({os.path.basename(latest_log)}):")
                try:
                    with open(latest_log, 'r') as f:
                        lines = f.readlines()
                        for line in lines[-10:]:  # Last 10 lines
                            print(f"      {line.strip()}")
                except:
                    pass
    else:
        print("\n  No training runs yet")
else:
    print("⚠️ No training logs found yet")
    print("   Start training first (see Test 9)")

print("\n" + "="*70)
print("After training completes, you can:")
print("  1. Download best checkpoint: kaggle_logs/.../best.pth")
print("  2. Evaluate on test set using deepfakebench/test.py")
print("  3. Use the model for inference")